**Legacy notebook.** Self-contained analysis code that predates the `src/mrvf` library and has not been ported to it. Kept for provenance and because it still produces figures in `results/`. Paths were updated to the `results/` layout; the next cell sets the working directory to the repository root, so run it from anywhere.

For the maintained pipeline see `notebooks/01_train_triple_regime.ipynb` and `notebooks/02_evaluate_rmse_vs_snr.ipynb`.

In [ ]:
import os
from pathlib import Path
# run from the repository root so ./results/... and ../subsamples resolve
_root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src" / "mrvf").is_dir())
os.chdir(_root)

# Notebook 2 (v5): In Vivo Inference — DL-4p vs T2-Conditioned

| Section | Content |
|---------|----------|
| §1 | Imports & GPU |
| §2 | Configuration |
| §3 | MAT loader |
| §4 | Model architecture + loading (Conv1DModel for both models) |
| §5 | Dictionary + GPU DM |
| §6 | Load in-vivo data |
| §7 | Helpers + T2 estimation (BC and ABC regimes) |
| §8 | Process all subjects |
| §9 | Save ROI statistics |
| §10 | Brain maps — DL-4p vs T2cond-BC vs T2cond-ABC |
| §11 | Gas challenge quantification |


## 1. Imports & GPU

In [ ]:
import os, re, json, time, glob
import numpy as np
import scipy.io as sio
import nibabel as nib
import matplotlib.pyplot as plt
import pandas as pd
import h5py
from scipy import stats as sci_stats

import torch
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


## 2. Configuration

In [ ]:
CONFIG = {
    # ── In vivo data ─────────────────────────────────────────────
    'images_mat': '../code/images.mat',
    'masks_dir':  '../GESFIDE_data/GES_ROI',

    # ── Models ───────────────────────────────────────────────────
    # v4 4-param model (baseline — best simulation performance)
    'model_dir':         './results/t2snr_results_v4/models',
    'model_4p_name':     't2snr_noisy_4param_v4.pt',
    # T2-conditioned 3-param model (v5)
    'model_t2cond_dir':  './results/t2snr_results_v5/models',
    'model_t2cond_name': 't2cond_film_v5.pt',

    # ── Dictionary ───────────────────────────────────────────────
    'dict_base_path':     '../subsamples/subsamples_v3',
    'param_path':         '../subsamples/subsamples_v3/QuasiRand_par_t2_200.mat',
    'noisefree_sig_path': '../subsamples/subsamples_v3/QuasiRand_t2_200.mat',
    'dict_key':  'Dico40_save',
    'param_key': 'par',

    # ── Parameter info (4-param model) ───────────────────────────
    'param_mins':     np.array([0.0,   0.0025,  1.0e-6,  0.050]),
    'param_maxs':     np.array([1.0,   0.15,   25.0e-6,  0.200]),
    'param_names_4p': ['SO2', 'CBV', 'R', 'T2'],
    'param_names_3p': ['SO2', 'CBV', 'R'],

    # ── GESFIDE sequence (for analytical T2 estimation) ──────────
    'n_echoes':     40,
    'n_fid':        14,       # echoes before 180° pulse
    'echo_spacing': 3e-3,     # seconds

    # ── Output ───────────────────────────────────────────────────
    'output_dir': './results/t2snr_results_v5/invivo',

    # ── Inference ────────────────────────────────────────────────
    'dl_batch_size': 8192,
    'dm_batch_size': 200,
    'baseline_condition': 'air',
}

os.makedirs(CONFIG['output_dir'], exist_ok=True)
print('Output dir:', CONFIG['output_dir'])


## 3. MAT File Loader

In [ ]:
def load_mat(path, key):
    '''Handles both MATLAB v5 and v7.3/HDF5 .mat files.'''
    try:
        mat = sio.loadmat(path)
        if key in mat:
            return np.array(mat[key], dtype=np.float32)
        cands = [k for k in mat if not k.startswith('_')]
        print(f'  ⚠ Key "{key}" not found — using "{cands[0]}"')
        return np.array(mat[cands[0]], dtype=np.float32)
    except NotImplementedError:
        with h5py.File(path, 'r') as f:
            data = f[key][()] if key in f else f[
                next(k for k in f if not k.startswith('#'))][()]
            if data.ndim >= 2: data = data.T
            return np.array(data, dtype=np.float32)


def load_images_mat(path):
    try:
        return sio.loadmat(path)
    except NotImplementedError:
        out = {}
        with h5py.File(path, 'r') as f:
            for k in f.keys():
                if k.startswith('#'): continue
                arr = np.array(f[k][()])
                if arr.ndim >= 2: arr = arr.T
                out[k] = arr
        return out


print('✓ MAT loaders ready')


## 4. Model Architecture & Loading

Both models use `Conv1DModel` — the same architecture from v4.  
The T2-conditioned model simply has `in_dim=41` (signal + T2 scalar) and `n_outputs=3`.


In [ ]:
class Clamp01(nn.Module):
    def forward(self, x): return x.clamp(0.0, 1.0)


class Conv1DModel(nn.Module):
    def __init__(self, n_outputs=4, dropout=0.3, in_dim=40):
        super().__init__()
        self.conv_path = nn.Sequential(
            nn.Conv1d(1, 32,  kernel_size=7, padding=3),
            nn.BatchNorm1d(32),  nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(dropout),
            nn.Conv1d(32, 64,  kernel_size=5, padding=2),
            nn.BatchNorm1d(64),  nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(dropout),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256), nn.ReLU(),
        )  # 40 → 20 → 10 → 5  conv_out = 256 * 5 = 1280
        conv_out = 256 * 5
        extra = in_dim - 40
        self.mlp = nn.Sequential(
            nn.Linear(conv_out + extra, 2048), nn.BatchNorm1d(2048), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(2048, 1024),             nn.BatchNorm1d(1024), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(1024, 512),              nn.BatchNorm1d(512),  nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(512,  256),              nn.BatchNorm1d(256),  nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256,  n_outputs),
        )
        self.out_act = Clamp01()

    def forward(self, x):
        echo  = x[:, :40].unsqueeze(1)
        extra = x[:, 40:]
        c = self.conv_path(echo).flatten(1)
        return self.out_act(self.mlp(torch.cat([c, extra], dim=1)))


def load_conv1d_model(path):
    '''Load Conv1DModel from checkpoint — infers in_dim and n_outputs automatically.'''
    ckpt = torch.load(path, map_location=device, weights_only=False)
    n_out  = ckpt['n_outputs']
    # Detect in_dim from first MLP weight shape
    mlp_in        = ckpt['model_state_dict']['mlp.0.weight'].shape[1]
    conv_keys      = [k for k in ckpt['model_state_dict']
                      if 'conv_path' in k and 'weight' in k and 'BatchNorm' not in k]
    last_ch        = ckpt['model_state_dict'][sorted(conv_keys)[-1]].shape[0]
    spatial        = mlp_in // last_ch
    extra          = mlp_in - last_ch * spatial
    in_dim         = 40 + extra
    dropout        = ckpt.get('dropout', 0.3)
    model = Conv1DModel(n_outputs=n_out, dropout=dropout, in_dim=in_dim).to(device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    print(f'  ✓ {os.path.basename(path)}: n_outputs={n_out}  in_dim={in_dim}')
    return model, ckpt


# ── Load both models ─────────────────────────────────────────────────────────
print('Loading DL-4p model (v4)...')
model_4p, ckpt_4p = load_conv1d_model(
    os.path.join(CONFIG['model_dir'], CONFIG['model_4p_name']))

print('Loading T2-conditioned model (v5)...')
model_t2cond, ckpt_t2c = load_conv1d_model(
    os.path.join(CONFIG['model_t2cond_dir'], CONFIG['model_t2cond_name']))

# T2 scaling pulled from checkpoint — ensures training/inference match
T2_MIN = float(ckpt_t2c['t2_min'])
T2_MAX = float(ckpt_t2c['t2_max'])
PARAM_MINS_3P = np.array(ckpt_t2c['param_mins'])[:3]
PARAM_MAXS_3P = np.array(ckpt_t2c['param_maxs'])[:3]
print(f'T2 scaling from checkpoint: [{T2_MIN*1000:.0f}, {T2_MAX*1000:.0f}] ms')


## 5. Load Dictionary + GPU Dictionary Matching

In [ ]:
print('Loading noise-free dictionary...')
dm_sigs_raw = load_mat(CONFIG['noisefree_sig_path'], CONFIG['dict_key'])
dm_pars_raw = load_mat(CONFIG['param_path'], CONFIG['param_key'])[:, :4]

n_dm = min(len(dm_sigs_raw), len(dm_pars_raw))
dm_sigs_raw, dm_pars_raw = dm_sigs_raw[:n_dm], dm_pars_raw[:n_dm]

# Filter to physiological range
mask_dm = np.ones(n_dm, dtype=bool)
for i in range(4):
    mask_dm &= ((dm_pars_raw[:, i] >= CONFIG['param_mins'][i]) &
                (dm_pars_raw[:, i] <= CONFIG['param_maxs'][i]))
valid_dm = np.all(np.isfinite(dm_sigs_raw), axis=1)
dm_sigs_raw = dm_sigs_raw[mask_dm & valid_dm]
dm_pars_raw = dm_pars_raw[mask_dm & valid_dm]

norms_dm    = np.linalg.norm(np.abs(dm_sigs_raw), axis=1, keepdims=True)
dict_norm   = dm_sigs_raw / np.maximum(norms_dm, 1e-12)
dict_t_gpu  = torch.tensor(dict_norm, dtype=torch.float16).to(device)
print(f'Dictionary on GPU: {dict_t_gpu.shape}  '
      f'({dict_t_gpu.element_size()*dict_t_gpu.nelement()/1e6:.0f} MB)')


def gpu_dm_predict(test_sigs_norm, proc_idx, n_voxels):
    batch_size = CONFIG['dm_batch_size']
    y_pred = np.full((n_voxels, 4), np.nan, dtype=np.float32)
    t0 = time.time(); i = 0
    while i < len(proc_idx):
        sl = proc_idx[i:i + batch_size]
        try:
            batch = torch.tensor(test_sigs_norm[sl], dtype=torch.float16).to(device)
            best  = torch.mm(batch, dict_t_gpu.T).argmax(dim=1).cpu().numpy()
            y_pred[sl] = dm_pars_raw[best]
            del batch; torch.cuda.empty_cache(); i += batch_size
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            batch_size = max(50, batch_size // 2)
            print(f'    OOM — batch_size → {batch_size}')
    print(f'    GPU DM: {time.time()-t0:.1f}s  ({len(proc_idx):,} voxels)')
    return y_pred


print('✓ GPU DM ready')


## 6. Load In-Vivo Data

In [ ]:
print(f'Loading: {CONFIG["images_mat"]}')
mat = load_images_mat(CONFIG['images_mat'])
subj_keys = sorted([k for k in mat if k.lower().startswith('img_')])
print(f'Found {len(subj_keys)} scans:')
for k in subj_keys:
    print(f'  {k}: {np.asarray(mat[k]).shape}')


## 7. Helpers & Analytical T2 Estimation

Two T2 estimation regimes from Ni et al. (2015):
- **BC**: exponential fit to post-SE echoes only → cleanest T2 estimate
- **ABC**: uses both FID and post-SE slopes → `R2 = (R2*_A + R2*_B) / 2`

Both are tested at inference — fast to run, no retraining needed.


In [ ]:
def euclidean_norm(data):
    data = np.abs(data).astype(np.float32)
    norms = np.linalg.norm(data, axis=1, keepdims=True)
    return data / np.maximum(norms, 1e-12)

def params_inverse(scaled, mins, maxs):
    return scaled * (maxs - mins) + mins

def key_to_subject_id(img_key):
    m = re.match(r'img_(e\d+)', img_key, flags=re.IGNORECASE)
    return m.group(1).upper() if m else img_key

def key_to_condition(img_key):
    m = re.match(r'img_e\d+(.*)', img_key, flags=re.IGNORECASE)
    raw = m.group(1).lower().strip('_') if m else ''
    for tag, label in [('air','air'),('norm','air'),('hyper','hyper'),('hypo','hypo')]:
        if tag in raw: return label
    return raw or 'unknown'

def find_mask(subj_id):
    d = CONFIG['masks_dir']
    for sfx in ['', '_AIR', '_HYPER', '_HYPO', '_air', '_hyper', '_hypo']:
        for ext in ['.nii.gz', '.nii']:
            for sid in [subj_id, subj_id.lower()]:
                p = os.path.join(d, f'{sid}{sfx}_ROI{ext}')
                if os.path.exists(p): return p
    raise FileNotFoundError(f'No mask for {subj_id} in {d}')


# ── Vectorised log-linear slope per voxel ────────────────────────────────────
def _loglin_slope(t_vec, sig_mat):
    log_s  = np.log(np.maximum(np.abs(sig_mat), 1e-9)).astype(np.float64)
    t      = t_vec.astype(np.float64)
    t_c    = t - t.mean()
    log_s_c = log_s - log_s.mean(axis=1, keepdims=True)
    return (log_s_c * t_c[None, :]).sum(axis=1) / (t_c**2).sum()


def estimate_t2_BC(sig_raw):
    '''
    T2 from post-SE echoes only (regime B/C).
    S(dt) = S0 * exp(-dt/T2), dt = time after SE peak.
    '''
    n_fid = CONFIG['n_fid']          # 14
    dTE   = CONFIG['echo_spacing']    # 3e-3 s
    n_post = CONFIG['n_echoes'] - n_fid
    dt_BC  = np.arange(1, n_post + 1) * dTE
    slope  = _loglin_slope(dt_BC, sig_raw[:, n_fid:])
    T2_est = 1.0 / np.maximum(-slope, 1.0)
    return np.clip(T2_est, T2_MIN, T2_MAX).astype(np.float32)


def estimate_t2_ABC(sig_raw):
    '''
    T2 from FID + post-SE slopes (Ni et al. 2015, Eq. 3).
    R2 = (R2*_A + R2*_B) / 2  →  T2 = 1/R2
    '''
    n_fid  = CONFIG['n_fid']
    dTE    = CONFIG['echo_spacing']
    t_A    = np.arange(1, n_fid + 1) * dTE
    t_B    = np.arange(1, CONFIG['n_echoes'] - n_fid + 1) * dTE
    R2A    = np.maximum(-_loglin_slope(t_A, sig_raw[:, :n_fid]),    1.0)
    R2B    = np.maximum(-_loglin_slope(t_B, sig_raw[:, n_fid:]),    1.0)
    T2_est = 1.0 / np.maximum((R2A + R2B) / 2.0, 1.0)
    return np.clip(T2_est, T2_MIN, T2_MAX).astype(np.float32)


def build_t2cond_input(sig_norm, t2_est):
    '''
    Append scaled T2 scalar to L2-norm signal → (N, 41).
    T2 scaling uses T2_MIN/T2_MAX from the checkpoint.
    '''
    t2_scaled = np.clip((t2_est - T2_MIN) / (T2_MAX - T2_MIN), 0.0, 1.0)
    return np.concatenate([sig_norm, t2_scaled[:, None].astype(np.float32)], axis=1)


print('✓ Helpers and T2 estimation functions ready')


## 8. Process All Subjects

For each scan runs:
1. **DL-4p** — v4 model, 40-dim input, outputs SO2/CBV/R/T2
2. **DL-T2cond-BC** — v5 model, T2 from post-SE fit
3. **DL-T2cond-ABC** — v5 model, T2 from Ni et al. two-slope method
4. **DM** — GPU inner-product matching


In [ ]:
def run_dl(model, X_input, proc_idx, n_vox, n_out, label):
    '''Run DL model on proc_idx voxels. Returns (n_vox, n_out) physical array.'''
    scaled = np.full((n_vox, n_out), np.nan, dtype=np.float32)
    t0 = time.time()
    model.eval()
    with torch.no_grad():
        for start in range(0, proc_idx.size, CONFIG['dl_batch_size']):
            sl  = proc_idx[start:start + CONFIG['dl_batch_size']]
            xb  = torch.tensor(X_input[sl], dtype=torch.float32, device=device)
            scaled[sl] = model(xb).cpu().numpy()
            del xb
    print(f'    {label}: {time.time()-t0:.1f}s')
    return scaled


def process_subject(img_key):
    sig4d = np.asarray(mat[img_key], dtype=np.float32)
    if sig4d.ndim == 4 and sig4d.shape[0] == 40:
        sig4d = np.transpose(sig4d, (1, 2, 3, 0))
    H, W, S, T = sig4d.shape
    assert T == 40, f'Expected 40 echoes, got {T}'

    subj_id   = key_to_subject_id(img_key)
    condition = key_to_condition(img_key)

    mask_path = find_mask(subj_id)
    mask3d    = nib.load(mask_path).get_fdata()
    if mask3d.ndim == 4: mask3d = mask3d[..., 0]
    assert mask3d.shape == (H, W, S), f'Mask shape mismatch'

    X        = sig4d.reshape(-1, T)        # (N_vox, 40)  raw
    X_norm   = euclidean_norm(X)           # (N_vox, 40)  L2-norm
    n_vox    = X.shape[0]
    proc_idx = np.where(
        np.isfinite(X_norm).all(axis=1) & mask3d.flatten().astype(bool)
    )[0]
    print(f'  {img_key}: {proc_idx.size:,} voxels  (cond={condition})')

    # ── 1. DL-4p ──────────────────────────────────────────────────
    sc_4p = run_dl(model_4p, X_norm, proc_idx, n_vox, 4, 'DL-4p')
    phys_4p = np.full((n_vox, 4), np.nan, dtype=np.float32)
    phys_4p[proc_idx] = params_inverse(
        sc_4p[proc_idx], CONFIG['param_mins'], CONFIG['param_maxs'])

    # ── 2. Analytical T2 estimation ────────────────────────────────
    T2_BC  = np.full(n_vox, np.nan, dtype=np.float32)
    T2_ABC = np.full(n_vox, np.nan, dtype=np.float32)
    T2_BC[proc_idx]  = estimate_t2_BC(X[proc_idx])
    T2_ABC[proc_idx] = estimate_t2_ABC(X[proc_idx])
    print(f'    T2_BC  mean={np.nanmean(T2_BC[proc_idx])*1000:.1f} ms  '
          f'std={np.nanstd(T2_BC[proc_idx])*1000:.1f}')
    print(f'    T2_ABC mean={np.nanmean(T2_ABC[proc_idx])*1000:.1f} ms  '
          f'std={np.nanstd(T2_ABC[proc_idx])*1000:.1f}')

    # ── 3. T2-conditioned DL (BC and ABC) ─────────────────────────
    X_bc  = build_t2cond_input(X_norm, T2_BC)   # (N_vox, 41)
    X_abc = build_t2cond_input(X_norm, T2_ABC)

    sc_bc  = run_dl(model_t2cond, X_bc,  proc_idx, n_vox, 3, 'T2cond-BC')
    sc_abc = run_dl(model_t2cond, X_abc, proc_idx, n_vox, 3, 'T2cond-ABC')

    phys_bc  = np.full((n_vox, 3), np.nan, dtype=np.float32)
    phys_abc = np.full((n_vox, 3), np.nan, dtype=np.float32)
    phys_bc[proc_idx]  = params_inverse(sc_bc[proc_idx],  PARAM_MINS_3P, PARAM_MAXS_3P)
    phys_abc[proc_idx] = params_inverse(sc_abc[proc_idx], PARAM_MINS_3P, PARAM_MAXS_3P)

    # ── 4. DM ──────────────────────────────────────────────────────
    phys_dm = gpu_dm_predict(X_norm, proc_idx, n_vox)

    return {
        'subject_id': subj_id, 'condition': condition,
        'shape': (H, W, S), 'mask': mask3d, 'mask_file': mask_path,
        'DL_4p':       phys_4p.reshape(H, W, S, 4),
        'DL_T2cond_BC':  phys_bc.reshape(H, W, S, 3),
        'DL_T2cond_ABC': phys_abc.reshape(H, W, S, 3),
        'DM':          phys_dm.reshape(H, W, S, 4),
        'T2_map_BC':   T2_BC.reshape(H, W, S),
        'T2_map_ABC':  T2_ABC.reshape(H, W, S),
    }


all_results   = {}
all_roi_stats = []

for img_key in subj_keys:
    print(f'\n{"─"*55}')
    try:
        res = process_subject(img_key)
    except Exception as e:
        print(f'  ⚠ SKIPPED {img_key}: {e}')
        continue

    subj_id   = res['subject_id']
    condition = res['condition']
    mask3d    = res['mask']

    # Save per-subject maps
    out_path = os.path.join(CONFIG['output_dir'], f'{img_key}_maps_v5.mat')
    sio.savemat(out_path, {
        'DL_4param':      res['DL_4p'],
        'DM_4param':      res['DM'],
        'DL_T2cond_BC':   res['DL_T2cond_BC'],
        'DL_T2cond_ABC':  res['DL_T2cond_ABC'],
        'T2_est_BC':      res['T2_map_BC'],
        'T2_est_ABC':     res['T2_map_ABC'],
        'mask':           mask3d,
        'subject_key':    img_key,
        'param_names_4p': np.array(CONFIG['param_names_4p'], dtype=object),
        'param_names_3p': np.array(CONFIG['param_names_3p'], dtype=object),
    })
    all_results[img_key] = res

    # ROI stats
    gm_flat = (mask3d > 0).flatten()
    for method, maps, pnames, scales in [
        ('DL_4p',       res['DL_4p'],        ['SO2','CBV','R','T2'], [100,100,1e6,1000]),
        ('DM',          res['DM'],            ['SO2','CBV','R','T2'], [100,100,1e6,1000]),
        ('T2cond_BC',   res['DL_T2cond_BC'],  ['SO2','CBV','R'],      [100,100,1e6]),
        ('T2cond_ABC',  res['DL_T2cond_ABC'], ['SO2','CBV','R'],      [100,100,1e6]),
    ]:
        for pidx, (pname, scale) in enumerate(zip(pnames, scales)):
            vals = maps[..., pidx].flatten()[gm_flat] * scale
            vals = vals[np.isfinite(vals)]
            if not len(vals): continue
            all_roi_stats.append({
                'subject': subj_id, 'condition': condition,
                'method': method, 'region': 'GM_mask',
                'parameter': pname,
                'unit': '%' if pname in ['SO2','CBV'] else ('µm' if pname=='R' else 'ms'),
                'mean': float(np.nanmean(vals)),
                'std':  float(np.nanstd(vals)),
                'n_voxels': len(vals),
            })

print(f'\n✓ Done. Processed {len(all_results)} scans.')


## 9. Save ROI Statistics

In [ ]:
df = pd.DataFrame(all_roi_stats)
csv_path = os.path.join(CONFIG['output_dir'], 'roi_statistics_v5.csv')
df.to_csv(csv_path, index=False)
print(f'Saved: {csv_path}  shape={df.shape}')

if len(df) > 0:
    grp = df.groupby(['method','region','parameter','condition']).agg(
        mean=('mean','mean'), std=('mean','std'), n_subj=('subject','nunique')
    ).reset_index()
    grp.to_csv(os.path.join(CONFIG['output_dir'], 'roi_group_summary_v5.csv'), index=False)
    print(grp.to_string(index=False))


## 10. Brain Maps — DL-4p vs T2cond-BC vs T2cond-ABC

3-param comparison (SO2, CBV, R): 3 rows × 3 methods + T2 estimate QC.


In [ ]:
baseline_key = next(
    (k for k in subj_keys
     if CONFIG['baseline_condition'] in key_to_condition(k) and k in all_results),
    next((k for k in subj_keys if k in all_results), None)
)

if baseline_key:
    res    = all_results[baseline_key]
    mask3d = res['mask']
    sl     = int(mask3d.sum(axis=(0,1)).argmax())
    print(f'Subject: {baseline_key}  cond={key_to_condition(baseline_key)}  slice={sl}')

    param_vis = [
        ('SO₂ (%)',  0, 100,  (50, 100), 'hot'),
        ('CBV (%)',  1, 100,  (2,   8),  'turbo'),
        ('R (µm)',   2, 1e6,  (6,  22),  'pink'),
    ]
    method_list = [
        ('DL-4p',        res['DL_4p'],        4),
        ('T2cond (BC)',  res['DL_T2cond_BC'],  3),
        ('T2cond (ABC)', res['DL_T2cond_ABC'], 3),
        ('DM',           res['DM'],            4),
    ]

    fig, axes = plt.subplots(3, 4, figsize=(13, 9))
    fig.suptitle(f'{baseline_key} | Slice {sl} | {key_to_condition(baseline_key)}',
                 fontsize=11, fontweight='bold')
    fig.patch.set_facecolor('black')

    for row, (plabel, pidx, scale, (vmin, vmax), cmap_n) in enumerate(param_vis):
        cmap_obj = plt.cm.get_cmap(cmap_n).copy()
        cmap_obj.set_bad('black')
        for col, (mname, maps, n_out) in enumerate(method_list):
            ax = axes[row, col]
            ax.set_facecolor('black')
            if pidx < n_out:
                img = maps[:, :, sl, pidx] * scale
                img[mask3d[:, :, sl] == 0] = np.nan
                im = ax.imshow(np.rot90(img), cmap=cmap_obj,
                               vmin=vmin, vmax=vmax, interpolation='nearest')
                if col == 3:
                    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
                    cb.ax.tick_params(labelsize=6, colors='white')
            ax.axis('off')
            if row == 0: ax.set_title(mname, fontsize=9, color='white', pad=3)
            if col == 0:
                ax.set_ylabel(plabel, fontsize=8, color='white')
                ax.yaxis.set_visible(True); ax.set_yticks([])

    plt.tight_layout()
    fig_path = os.path.join(CONFIG['output_dir'], f'maps_{baseline_key}_v5.png')
    plt.savefig(fig_path, dpi=200, bbox_inches='tight', facecolor='black')
    plt.show()

    # ── T2 estimate QC ───────────────────────────────────────────────────────
    fig2, axes2 = plt.subplots(1, 2, figsize=(8, 3.5))
    fig2.suptitle(f'Analytical T2 estimates — {baseline_key} slice {sl}')
    for ax, (lbl, t2map, color) in zip(axes2, [
            ('BC regime',  res['T2_map_BC'],  'purple'),
            ('ABC regime', res['T2_map_ABC'], 'crimson'),
    ]):
        img = t2map[:, :, sl] * 1000
        img[mask3d[:, :, sl] == 0] = np.nan
        im = ax.imshow(np.rot90(img), cmap='bone', vmin=40, vmax=150,
                       interpolation='nearest')
        ax.set_title(f'T2 {lbl} (ms)', fontsize=9)
        ax.axis('off')
        plt.colorbar(im, ax=ax)
        # Distribution
        vals = t2map[mask3d > 0] * 1000
        vals = vals[np.isfinite(vals)]
        ax.set_xlabel(f'mean={vals.mean():.0f}ms  std={vals.std():.0f}ms',
                      color=color, fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG['output_dir'], f't2_qc_{baseline_key}_v5.png'), dpi=150)
    plt.show()


## 11. Gas Challenge Quantification (All Methods)

In [ ]:
df = pd.DataFrame(all_roi_stats)
if len(df) == 0:
    print('No ROI stats — run §8 first')
else:
    conds_found = df['condition'].unique()
    cond_order  = [c for c in ['air','hyper','hypo'] if c in conds_found]
    cond_colors = {'air':'#5B9BD5', 'hyper':'#ED7D31', 'hypo':'#70AD47'}
    cond_labels = {'air':'Normoxia', 'hyper':'Hyperoxia', 'hypo':'Hypoxia'}

    method_styles = [
        ('DL_4p',      'DL-4p',         1.0,  ''),
        ('T2cond_BC',  'T2cond (BC)',    0.75, '//'),
        ('T2cond_ABC', 'T2cond (ABC)',   0.75, 'xx'),
        ('DM',         'DM',             0.5,  '..'),
    ]
    param_info = [('SO2','SO₂ (%)',100), ('CBV','CBV (%)',100), ('R','R (µm)',1e6)]

    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    fig.suptitle('GM Parameters — All Methods Across Gas Conditions',
                 fontsize=11, fontweight='bold')

    n_methods = len(method_styles)
    width = 0.8 / n_methods

    for ax, (pname, ylabel, scale) in zip(axes, param_info):
        x = np.arange(len(cond_order))
        for mi, (mkey, mlabel, alpha, hatch) in enumerate(method_styles):
            means, sems = [], []
            for cond in cond_order:
                sub = df[(df.parameter==pname) & (df.method==mkey)
                         & (df.condition==cond)]
                means.append(sub['mean'].mean() if len(sub) else np.nan)
                sems.append(sub['mean'].std() / max(np.sqrt(len(sub)), 1)
                            if len(sub) else 0)
            offset = (mi - (n_methods-1)/2) * width
            ax.bar(x + offset, means, width * 0.9,
                   color=[cond_colors.get(c,'gray') for c in cond_order],
                   alpha=alpha, hatch=hatch, edgecolor='black', lw=0.5,
                   label=mlabel)
            ax.errorbar(x + offset, means, yerr=sems, fmt='none',
                        color='black', capsize=3)

            # Significance vs air
            for ci, cond in enumerate(cond_order):
                if cond == CONFIG['baseline_condition']: continue
                bv = df[(df.parameter==pname)&(df.method==mkey)
                        &(df.condition==CONFIG['baseline_condition'])]['mean'].values
                cv = df[(df.parameter==pname)&(df.method==mkey)
                        &(df.condition==cond)]['mean'].values
                if len(bv) >= 2 and len(cv) >= 2:
                    _, p = sci_stats.ttest_ind(bv, cv)
                    if p < 0.05:
                        ymax = means[ci] + sems[ci] if not np.isnan(means[ci]) else 0
                        ax.text(x[ci]+offset, ymax*1.06, '*',
                                ha='center', fontsize=11, color='red')

        ax.set_xticks(x)
        ax.set_xticklabels([cond_labels.get(c,c) for c in cond_order],
                            rotation=15, ha='right')
        ax.set_ylabel(ylabel); ax.set_title(pname)
        ax.grid(True, axis='y', alpha=0.3)
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
        if pname == 'SO2':
            ax.legend(fontsize=7, bbox_to_anchor=(1.0, 1.0))

    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG['output_dir'], 'gas_challenge_v5.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
